# MethylLlama — Age Prediction & Fine-tuning

This notebook explains the full pipeline from WCED pretraining to age prediction
and shows the paper's main result: WCED pretraining vs random initialisation.

**Paper result (test set)**

| Model | R² | MedAE | MAE |
|---|---|---|---|
| WCED pretrained (job 44895876) | **0.905** | **3.65 yr** | 5.46 yr |
| Random init (job 44981091) | 0.526 | 8.91 yr | 13.06 yr |

Same architecture (256D × 4L × 4H), same data (21k CpGs, 10,988 samples),
same fine-tuning recipe — only the initialisation differs.

In [ ]:
import sys
from pathlib import Path

repo_root = Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. WCED pretraining concept

**Whole-Cell Expression Decoder (WCED)** is a self-supervised pretraining objective:

```
50 % of CpG β-values  →  MethylLlama encoder  →  CLS token
                                                    ↓
                                              WCED decoder  →  reconstruct ALL CpG β-values
                                                    +
                                              InfoNCE loss (two random views per sample)
                                                    +
                                              auxiliary age head
```

The encoder must compress the full methylation profile into the 256-D CLS vector
to reconstruct the held-out 50 % — forcing globally informative representations.

**Contrastive term (InfoNCE)**: two independent 50 % subsets of the same sample
form a positive pair. The encoder learns representations that are invariant to
which subset is shown, making them more robust.

In [ ]:
# Visualise the pretraining learning curve (schematic)
pretrain_epochs = np.arange(1, 99)

recon_loss = 0.35 * np.exp(-pretrain_epochs / 30) + 0.13 + 0.01 * np.random.default_rng(0).standard_normal(98)
recon_loss = np.clip(recon_loss, 0.10, 0.36)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(pretrain_epochs, recon_loss, lw=2)
ax.axvline(98, color="red", ls="--", label="Best ckpt (epoch 98, val_loss=0.0059)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss")
ax.set_title("WCED pretraining convergence (schematic)")
ax.legend()
plt.tight_layout()
plt.savefig("wced_convergence.png", dpi=150)
plt.show()

## 2. Fine-tuning protocol

```
Pretrained encoder (frozen for first 10 epochs)
  → CLS pooler_output  (256-D)
  → Linear(256 → 256) → ReLU → Dropout(0)
  → Linear(256 → 1)   → predicted age

Loss: Huber (δ = 5 yr / age_std)
      + recon_weight=0 × reconstruction  (no regulariser in V5)
```

**Key V5 hyperparameters**:
- LR: 1e-4 (head), 2e-5 (encoder after epoch 10 unfreeze)
- Effective batch size: 32 × 4 accum = 128
- Early stopping: patience 100
- Max epochs: 300 (converged at ~140)

## 3. Code walkthrough

The actual fine-tuning requires a cluster. This section walks through the code
path so you can understand what happens under the hood.

In [ ]:
# ── Step 1: Load pretrained encoder ──────────────────────────────────────────
# (Don't run — needs cluster and GPU)

code = """
import torch
from bmfm_methylation.llama.model import MethylLlamaConfig
from bmfm_methylation.llama.wced_llama import WCEDLlamaModule

ckpt_path = "outputs/.../epoch=98-val_loss=0.0059.ckpt"
ckpt      = torch.load(ckpt_path, map_location="cpu", weights_only=False)

vocab_size   = ckpt["state_dict"]["encoder.embeddings.cpg_sites_embeddings.weight"].shape[0]
model_config = MethylLlamaConfig(
    hidden_size=256, num_hidden_layers=4, num_attention_heads=4,
    intermediate_size=320, vocab_size=vocab_size,
)
pretrain_module = WCEDLlamaModule(
    model_config=model_config,
    vocab_size=ckpt["hyper_parameters"]["vocab_size"],
)
pretrain_module.load_state_dict(ckpt["state_dict"])
encoder = pretrain_module.encoder
"""
print(code)

In [ ]:
# ── Step 2: Build fine-tuning module ─────────────────────────────────────────

code = """
import pytorch_lightning as pl
import torch.nn as nn
from torchmetrics import R2Score, MeanAbsoluteError

class MethylationAgeRegressor(pl.LightningModule):
    def __init__(self, encoder, hidden=256, lr_head=1e-4, lr_encoder=2e-5):
        super().__init__()
        self.encoder  = encoder
        self.age_head = nn.Sequential(
            nn.Linear(256, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
        self.lr_head    = lr_head
        self.lr_encoder = lr_encoder
        self.loss_fn = nn.HuberLoss(delta=5.0)
        self.r2  = R2Score()
        self.mae = MeanAbsoluteError()

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.age_head(out.pooler_output).squeeze(-1)

    def training_step(self, batch, _):
        pred = self(batch["input_ids"], batch["attention_mask"])
        loss = self.loss_fn(pred, batch["ages"])
        self.log("train/loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW([
            {"params": self.age_head.parameters(), "lr": self.lr_head},
            {"params": self.encoder.parameters(),  "lr": self.lr_encoder},
        ], weight_decay=0.01)


model = MethylationAgeRegressor(encoder)
# Freeze encoder for first 10 epochs, then unfreeze
for p in model.encoder.parameters():
    p.requires_grad = False
"""
print(code)

## 4. Paper results

### 4a. WCED pretrained vs random initialisation

In [ ]:
models  = ["Random init\n(job 44981091)", "WCED pretrained\n(job 44895876)"]
r2      = [0.526, 0.905]
medae   = [8.91, 3.65]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ["#d62728", "#2ca02c"]

bars = axes[0].bar(models, r2, color=colors, width=0.4, edgecolor="k", linewidth=0.7)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Test R²", fontsize=12)
axes[0].set_title("Explained variance (R²)\nhigher is better", fontsize=11)
for bar, val in zip(bars, r2):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

bars2 = axes[1].bar(models, medae, color=colors, width=0.4, edgecolor="k", linewidth=0.7)
axes[1].set_ylabel("Test MedAE (years)", fontsize=12)
axes[1].set_title("Median absolute error\nlower is better", fontsize=11)
for bar, val in zip(bars2, medae):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.15,
                 f"{val:.2f} yr", ha="center", va="bottom", fontsize=11, fontweight="bold")

fig.suptitle("WCED Pretraining vs Random Initialisation\n"
             "256D × 4L × 4H MethylLlama, 21k CpGs, same fine-tuning recipe",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("wced_vs_random.png", dpi=150)
plt.show()
print("Saved: wced_vs_random.png")

### 4b. Summary

In [ ]:
print("=" * 55)
print("  MethylLlama — Paper Results (test set)")
print("=" * 55)
print(f"{'Model':<30} {'R²':>6}  {'MedAE':>8}  {'MAE':>8}")
print("-" * 55)
print(f"{'WCED pretrained':<30} {'0.905':>6}  {'3.65 yr':>8}  {'5.46 yr':>8}")
print(f"{'Random init':<30} {'0.526':>6}  {'8.91 yr':>8}  {'13.06 yr':>8}")
print("-" * 55)
print(f"{'Improvement (WCED / Random)':<30} {'1.72×':>6}  {'2.44×':>8}  {'2.39×':>8}")
print("=" * 55)

## 5. Running fine-tuning on the cluster

```bash
# Pretrained encoder → fine-tune on 21k CpG dataset
sbatch scripts/llama/finetune_llama_small_v5.sh

# Random initialisation ablation
sbatch scripts/llama/finetune_llama_random_init.sh
```

Both scripts configure:
- 21k CpG dataset: `altumage_21k_3way.h5ad`
- Batch 32 × accum 4 = 128 effective
- Huber loss, CLS pooling, early stop patience 100

Monitor training on WandB project `finetune-llama-small`.